In [ ]:
"""
QUICKBITE PRODUCT ANALYTICS PLATFORM
Phase 2 - Notebook 3: RFM Segmentation Analysis
==================================================================
Purpose: Segment users using Recency, Frequency, Monetary (RFM) analysis
to identify high-value segments, at-risk users, and personalization
opportunities.

Key Questions:
1. What are the natural user segments based on purchase behavior?
2. Which segments are most valuable and should be prioritized?
3. Which segments are at risk of churning?
4. What personalized strategies should be applied to each segment?
5. How do segments differ across channels and cities?

Author: Senior Product Analytics Team
Date: 2026-07-28
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
print("="*80)
print("QUICKBITE RFM SEGMENTATION ANALYSIS")
print("="*80)

---------------------------------------------------------------------
1. LOAD CLEANED DATA
---------------------------------------------------------------------

In [ ]:
print("\n📂 Loading cleaned data...")

In [ ]:
users = pd.read_csv('../outputs/cleaned_data/users_cleaned.csv')
orders = pd.read_csv('../outputs/cleaned_data/orders_cleaned.csv')
order_items = pd.read_csv('../outputs/cleaned_data/order_items_cleaned.csv')
cities = pd.read_csv('../data/cities.csv')

In [ ]:
# Convert dates
users['signup_date'] = pd.to_datetime(users['signup_date'])
orders['order_placed_at'] = pd.to_datetime(orders['order_placed_at'])

In [ ]:
# Filter delivered orders
delivered_orders = orders[orders['order_status'] == 'delivered']

In [ ]:
print(f"✅ Loaded {len(users):,} users")
print(f"✅ Loaded {len(delivered_orders):,} delivered orders")

---------------------------------------------------------------------
2. CALCULATE RFM METRICS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CALCULATING RFM METRICS")
print("="*80)

In [ ]:
# Reference date for recency calculation
reference_date = pd.Timestamp.now()

In [ ]:
# Calculate RFM metrics per user
rfm_data = []

In [ ]:
for user_id in delivered_orders['user_id'].unique():
    user_orders = delivered_orders[delivered_orders['user_id'] == user_id]
    
    # Recency: days since last order
    last_order_date = user_orders['order_placed_at'].max()
    recency = (reference_date - last_order_date).days
    
    # Frequency: number of orders
    frequency = len(user_orders)
    
    # Monetary: total spend
    monetary = user_orders['total_amount'].sum()
    
    # Additional metrics for richer segmentation
    avg_order_value = user_orders['total_amount'].mean()
    first_order_date = user_orders['order_placed_at'].min()
    customer_lifetime_days = (last_order_date - first_order_date).days if frequency > 1 else 0
    avg_days_between_orders = customer_lifetime_days / (frequency - 1) if frequency > 1 else None
    
    rfm_data.append({
        'user_id': user_id,
        'recency': recency,
        'frequency': frequency,
        'monetary': monetary,
        'avg_order_value': avg_order_value,
        'first_order_date': first_order_date,
        'last_order_date': last_order_date,
        'customer_lifetime_days': customer_lifetime_days,
        'avg_days_between_orders': avg_days_between_orders
    })

In [ ]:
rfm_df = pd.DataFrame(rfm_data)

In [ ]:
# Add user attributes
rfm_df = rfm_df.merge(
    users[['user_id', 'acquisition_channel', 'is_premium_member', 'city_id', 
           'age_band', 'device_type', 'signup_date']],
    on='user_id', how='left'
)

In [ ]:
# Add city names
rfm_df = rfm_df.merge(cities[['city_id', 'city_name', 'tier']], on='city_id', how='left')

In [ ]:
print(f"✅ RFM metrics calculated for {len(rfm_df):,} users")

In [ ]:
# Display RFM distribution
print("\n📊 RFM Distribution Statistics:")
print(rfm_df[['recency', 'frequency', 'monetary']].describe())

---------------------------------------------------------------------
3. RFM SCORING (Classic RFM Quartile Method)
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("RFM SCORING (Quartile Method)")
print("="*80)

In [ ]:
def assign_rfm_score(df, column, ascending=True):
    """
    Assign RFM scores using quartile method.
    R: Lower is better (more recent)
    F: Higher is better (more frequent)
    M: Higher is better (more spend)
    """
    if ascending:
        # For Recency: lower is better
        labels = [4, 3, 2, 1]  # 4 = best (most recent)
    else:
        # For Frequency & Monetary: higher is better
        labels = [1, 2, 3, 4]  # 4 = best (highest)
    
    # Use qcut with 4 equal bins
    df[f'{column}_score'] = pd.qcut(
        df[column], 
        q=4, 
        labels=labels, 
        duplicates='drop'
    ).astype(int)
    
    return df

In [ ]:
# Assign scores
rfm_df = assign_rfm_score(rfm_df, 'recency', ascending=True)
rfm_df = assign_rfm_score(rfm_df, 'frequency', ascending=False)
rfm_df = assign_rfm_score(rfm_df, 'monetary', ascending=False)

In [ ]:
# Calculate combined RFM score
rfm_df['rfm_score'] = rfm_df['recency_score'] * 100 + rfm_df['frequency_score'] * 10 + rfm_df['monetary_score']

In [ ]:
print("\n📊 RFM Score Distribution:")
print(rfm_df['rfm_score'].value_counts().sort_index().head(10))

---------------------------------------------------------------------
4. SEGMENT ASSIGNMENT (Classic RFM Segments)
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SEGMENT ASSIGNMENT")
print("="*80)

In [ ]:
def assign_segment(row):
    """Assign classic RFM segments"""
    r = row['recency_score']
    f = row['frequency_score']
    m = row['monetary_score']
    
    # High Value Segments
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 4 and m >= 3:
        return 'Loyal Customers'
    elif r >= 3 and f >= 3 and m >= 4:
        return 'Big Spenders'
    
    # Mid Value Segments
    elif r >= 2 and f >= 3 and m >= 3:
        return 'Potential Loyalists'
    elif r >= 3 and f >= 2 and m >= 2:
        return 'Recent Users'
    elif r >= 4 and f >= 2 and m >= 2:
        return 'New Customers'
    
    # At Risk Segments
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f >= 2 and m >= 2:
        return 'Dormant'
    elif r <= 1 and f >= 1 and m >= 1:
        return 'Lost'
    
    # Low Value
    elif r >= 2 and f <= 2 and m <= 2:
        return 'Low Value'
    
    # Fallback
    else:
        return 'Other'

In [ ]:
rfm_df['segment'] = rfm_df.apply(assign_segment, axis=1)

In [ ]:
print("\n📊 Segment Distribution:")
segment_counts = rfm_df['segment'].value_counts()
for segment, count in segment_counts.items():
    pct = count / len(rfm_df) * 100
    print(f"  • {segment}: {count:,} users ({pct:.1f}%)")

---------------------------------------------------------------------
5. SEGMENT METRICS & PROFILING
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SEGMENT METRICS & PROFILING")
print("="*80)

In [ ]:
# Calculate segment metrics
segment_metrics = rfm_df.groupby('segment').agg({
    'user_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': ['mean', 'sum'],
    'avg_order_value': 'mean',
    'customer_lifetime_days': 'mean'
}).round(2)

In [ ]:
segment_metrics.columns = ['users', 'avg_recency', 'avg_frequency', 'avg_monetary', 
                          'total_monetary', 'avg_order_value', 'avg_lifetime_days']

In [ ]:
segment_metrics = segment_metrics.sort_values('avg_monetary', ascending=False)

In [ ]:
print("\n📊 Segment Performance Metrics:")
print(segment_metrics)

In [ ]:
# Calculate % of total revenue and users
segment_metrics['pct_users'] = segment_metrics['users'] / segment_metrics['users'].sum() * 100
segment_metrics['pct_revenue'] = segment_metrics['total_monetary'] / segment_metrics['total_monetary'].sum() * 100

In [ ]:
print("\n📊 Revenue Concentration:")
for segment in segment_metrics.index:
    pct_users = segment_metrics.loc[segment, 'pct_users']
    pct_revenue = segment_metrics.loc[segment, 'pct_revenue']
    print(f"  • {segment}: {pct_users:.1f}% users → {pct_revenue:.1f}% revenue")

---------------------------------------------------------------------
6. VISUALIZE SEGMENTS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("VISUALIZING SEGMENTS")
print("="*80)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('RFM Segmentation Analysis', fontsize=16, fontweight='bold')

In [ ]:
# 6.1 Segment Distribution
ax = axes[0, 0]
segment_counts_sorted = segment_counts.sort_values(ascending=True)
colors = ['#2ecc71' if x in ['Champions', 'Loyal Customers'] 
          else '#f39c12' if x in ['Potential Loyalists', 'Recent Users', 'New Customers']
          else '#e74c3c' if x in ['At Risk', 'Dormant', 'Lost']
          else '#95a5a6' for x in segment_counts_sorted.index]
bars = ax.barh(segment_counts_sorted.index, segment_counts_sorted.values, color=colors, alpha=0.7)
ax.set_title('User Segment Distribution')
ax.set_xlabel('Number of Users')

In [ ]:
for bar in bars:
    width = bar.get_width()
    ax.text(width + 10, bar.get_y() + bar.get_height()/2, 
            f'{width:,}', ha='left', va='center', fontsize=9)

In [ ]:
# 6.2 Revenue Contribution by Segment
ax = axes[0, 1]
segment_revenue = segment_metrics[['pct_users', 'pct_revenue']].sort_values('pct_revenue', ascending=False)
x = np.arange(len(segment_revenue))
width = 0.35

In [ ]:
bars1 = ax.bar(x - width/2, segment_revenue['pct_users'], width, label='% Users', color='#3498db', alpha=0.7)
bars2 = ax.bar(x + width/2, segment_revenue['pct_revenue'], width, label='% Revenue', color='#e74c3c', alpha=0.7)

In [ ]:
ax.set_title('Revenue Concentration by Segment')
ax.set_xlabel('Segment')
ax.set_ylabel('Percentage (%)')
ax.set_xticks(x)
ax.set_xticklabels(segment_revenue.index, rotation=45, ha='right')
ax.legend()

In [ ]:
# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

In [ ]:
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

In [ ]:
# 6.3 RFM Score Heatmap
ax = axes[0, 2]
rfm_heatmap = rfm_df.groupby(['recency_score', 'frequency_score']).size().unstack(fill_value=0)
sns.heatmap(rfm_heatmap, ax=ax, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Users'})
ax.set_title('RFM Heatmap (R x F)')
ax.set_xlabel('Frequency Score')
ax.set_ylabel('Recency Score')

In [ ]:
# 6.4 Segment Metrics Comparison
ax = axes[1, 0]
metrics_to_plot = ['avg_frequency', 'avg_monetary', 'avg_order_value']
segment_metrics_scaled = segment_metrics[metrics_to_plot].copy()
for col in metrics_to_plot:
    segment_metrics_scaled[col] = (segment_metrics_scaled[col] - segment_metrics_scaled[col].min()) / \
                                   (segment_metrics_scaled[col].max() - segment_metrics_scaled[col].min())

In [ ]:
segment_metrics_scaled.plot(kind='bar', ax=ax)
ax.set_title('Segment Metrics (Normalized)')
ax.set_xlabel('Segment')
ax.set_ylabel('Normalized Value')
ax.tick_params(axis='x', rotation=45)
ax.legend(loc='upper right')

In [ ]:
# 6.5 Acquisition Channel Distribution by Segment
ax = axes[1, 1]
channel_segment = pd.crosstab(rfm_df['segment'], rfm_df['acquisition_channel'], normalize='index') * 100
channel_segment.plot(kind='barh', ax=ax, stacked=True)
ax.set_title('Acquisition Channel by Segment')
ax.set_xlabel('Percentage of Segment')
ax.set_ylabel('Segment')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
# 6.6 Premium Membership by Segment
ax = axes[1, 2]
premium_segment = rfm_df.groupby('segment')['is_premium_member'].mean() * 100
premium_segment = premium_segment.sort_values(ascending=False)
premium_segment.plot(kind='bar', ax=ax, color='#e67e22', alpha=0.7)
ax.set_title('Premium Membership by Segment')
ax.set_xlabel('Segment')
ax.set_ylabel('Premium Users (%)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
for i, v in enumerate(premium_segment.values):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/rfm_segments.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
7. SEGMENT-SPECIFIC RECOMMENDATIONS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SEGMENT-SPECIFIC RECOMMENDATIONS")
print("="*80)

In [ ]:
# Create segment recommendations
segment_recommendations = pd.DataFrame({
    'segment': [
        'Champions', 'Loyal Customers', 'Big Spenders', 
        'Potential Loyalists', 'Recent Users', 'New Customers',
        'At Risk', 'Dormant', 'Lost', 'Low Value', 'Other'
    ],
    'strategy': [
        'Reward loyalty, encourage referrals, cross-sell premium',
        'Maintain engagement, introduce new features first',
        'Upsell higher tiers, personalized offers',
        'Nurture to loyal, increase frequency with targeted campaigns',
        'Build habit, second-order incentives',
        'Great onboarding, fast value delivery',
        'Reactivate with win-back campaigns',
        'Re-engagement with strong incentives',
        'Re-evaluate if worth retaining',
        'Increase basket size, upselling',
        'Further analysis needed'
    ],
    'priority': [
        'P0', 'P0', 'P1',
        'P1', 'P1', 'P1',
        'P0', 'P1', 'P2',
        'P2', 'P2'
    ],
    'estimated_impact': [
        'High', 'High', 'Medium',
        'High', 'Medium', 'High',
        'High', 'Medium', 'Low',
        'Low', 'Low'
    ]
})

In [ ]:
print("\n📋 Segment Strategy Recommendations:")
print(segment_recommendations.to_string(index=False))

---------------------------------------------------------------------
8. K-MEANS CLUSTERING FOR SEGMENT VALIDATION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("K-MEANS CLUSTERING VALIDATION")
print("="*80)

In [ ]:
# Prepare data for clustering
cluster_data = rfm_df[['recency', 'frequency', 'monetary', 'avg_order_value']].copy()

In [ ]:
# Log transform for skewed variables
cluster_data['recency_log'] = np.log1p(cluster_data['recency'])
cluster_data['frequency_log'] = np.log1p(cluster_data['frequency'])
cluster_data['monetary_log'] = np.log1p(cluster_data['monetary'])

In [ ]:
# Scale features
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_data[['recency_log', 'frequency_log', 'monetary_log']])

In [ ]:
# Find optimal number of clusters
inertias = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(cluster_scaled)
    inertias.append(kmeans.inertia_)

In [ ]:
# Elbow plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(2, 11), inertias, marker='o', linewidth=2, color='#3498db')
ax.set_title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
ax.grid(True, alpha=0.3)

In [ ]:
# Mark elbow point (assuming k=4 or 5)
ax.axvline(5, color='red', linestyle='--', alpha=0.5, label='Optimal K=5')

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/rfm_elbow_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Apply K-means with optimal K
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm_df['cluster'] = kmeans.fit_predict(cluster_scaled)

In [ ]:
# Visualize clusters with PCA
pca = PCA(n_components=2)
cluster_pca = pca.fit_transform(cluster_scaled)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(cluster_pca[:, 0], cluster_pca[:, 1], 
                     c=rfm_df['cluster'], cmap='viridis', alpha=0.5, s=30)
ax.set_title('K-Means Clusters Visualization (PCA)', fontsize=14, fontweight='bold')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
plt.colorbar(scatter, label='Cluster')

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/rfm_kmeans_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Compare K-means clusters with RFM segments
cluster_segment_cross = pd.crosstab(rfm_df['cluster'], rfm_df['segment'])
print("\n📊 K-Means Clusters vs RFM Segments:")
print(cluster_segment_cross)

---------------------------------------------------------------------
9. SEGMENT SHIFT ANALYSIS (Loyalty Migration)
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("LOYALTY MIGRATION ANALYSIS")
print("="*80)

In [ ]:
# Define segment tiers for migration analysis
segment_tiers = {
    'Champions': 5,
    'Loyal Customers': 4,
    'Big Spenders': 4,
    'Potential Loyalists': 3,
    'Recent Users': 3,
    'New Customers': 3,
    'At Risk': 2,
    'Dormant': 1,
    'Lost': 0,
    'Low Value': 1,
    'Other': 2
}

In [ ]:
rfm_df['segment_tier'] = rfm_df['segment'].map(segment_tiers)

In [ ]:
# Calculate average tier by signup month
signup_month_tier = rfm_df.groupby(
    rfm_df['signup_date'].dt.to_period('M')
)['segment_tier'].mean().sort_index()

In [ ]:
# Calculate weighted average
signup_month_size = rfm_df.groupby(
    rfm_df['signup_date'].dt.to_period('M')
).size()

In [ ]:
# Visualize loyalty migration
fig, ax = plt.subplots(figsize=(14, 6))

In [ ]:
ax2 = ax.twinx()
signup_month_tier.plot(kind='line', ax=ax, color='#2ecc71', marker='o', linewidth=2)
signup_month_size.plot(kind='bar', ax=ax2, color='#3498db', alpha=0.3)

In [ ]:
ax.set_title('Loyalty Migration by Signup Cohort', fontsize=14, fontweight='bold')
ax.set_xlabel('Signup Month')
ax.set_ylabel('Average Segment Tier', color='#2ecc71')
ax2.set_ylabel('Cohort Size', color='#3498db')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/loyalty_migration.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
10. CITY SEGMENT DISTRIBUTION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CITY SEGMENT DISTRIBUTION")
print("="*80)

In [ ]:
# Calculate segment distribution by city
city_segments = pd.crosstab(rfm_df['city_name'], rfm_df['segment'], normalize='index') * 100

In [ ]:
# Focus on high-value segments
high_value_segments = ['Champions', 'Loyal Customers', 'Big Spenders']
city_high_value = city_segments[high_value_segments].sum(axis=1).sort_values(ascending=False)

In [ ]:
print("\n📊 Top 5 Cities by High-Value Segment %:")
for city, pct in city_high_value.head(5).items():
    print(f"  • {city}: {pct:.1f}% high-value users")

In [ ]:
print("\n📊 Bottom 5 Cities by High-Value Segment %:")
for city, pct in city_high_value.tail(5).items():
    print(f"  • {city}: {pct:.1f}% high-value users")

In [ ]:
# Visualize city segment distribution
fig, ax = plt.subplots(figsize=(14, 8))

In [ ]:
city_segments_top = city_segments.head(10)  # Top 10 cities
city_segments_top.plot(kind='barh', ax=ax, stacked=True)
ax.set_title('Segment Distribution by City (Top 10 Cities)', fontsize=14, fontweight='bold')
ax.set_xlabel('Percentage of Users')
ax.set_ylabel('City')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/city_segments.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
11. SEGMENT TRANSITION PROBABILITIES
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SEGMENT TRANSITION PROBABILITIES")
print("="*80)

Calculate segment transition probabilities (simplified)
For each segment, what % of users moved to higher/lower tier

In [ ]:
segment_migration = rfm_df.groupby('segment')['segment_tier'].agg(['count', 'mean'])
segment_migration['potential_upgrade'] = segment_migration['mean'].apply(
    lambda x: max(0, (5 - x) / 5 * 100)
)
segment_migration['potential_downgrade'] = segment_migration['mean'].apply(
    lambda x: min(100, (x / 5) * 100)
)

In [ ]:
print("\n📊 Segment Migration Potential:")
print(segment_migration.sort_values('mean', ascending=False))

In [ ]:
# Visualize segment lifecycle
fig, ax = plt.subplots(figsize=(10, 6))

In [ ]:
segments_order = ['Lost', 'Dormant', 'At Risk', 'Low Value', 'Recent Users', 
                  'New Customers', 'Potential Loyalists', 'Big Spenders', 
                  'Loyal Customers', 'Champions']

In [ ]:
segment_tier_order = [segment_tiers[s] for s in segments_order]
ax.plot(segment_tier_order, range(len(segments_order)), 'o-', linewidth=2, color='#3498db')
ax.set_title('Segment Loyalty Lifecycle', fontsize=14, fontweight='bold')
ax.set_xlabel('Segment Tier')
ax.set_ylabel('Segments (Low to High)')
ax.set_yticks(range(len(segments_order)))
ax.set_yticklabels(segments_order)
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/segment_lifecycle.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
12. SEGMENT-SPECIFIC METRICS DASHBOARD
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SEGMENT METRICS DASHBOARD")
print("="*80)

In [ ]:
# Create comprehensive segment dashboard
segment_dashboard = segment_metrics.copy()

In [ ]:
# Add calculated metrics
segment_dashboard['avg_recency_days'] = segment_dashboard['avg_recency']
segment_dashboard['avg_frequency_orders'] = segment_dashboard['avg_frequency']
segment_dashboard['avg_monetary_spend'] = segment_dashboard['avg_monetary']
segment_dashboard['avg_order_value'] = segment_dashboard['avg_order_value']
segment_dashboard['avg_lifetime_days'] = segment_dashboard['avg_lifetime_days']

In [ ]:
# Format for display
dashboard_display = segment_dashboard[['users', 'avg_recency_days', 'avg_frequency_orders',
                                       'avg_monetary_spend', 'avg_order_value', 
                                       'avg_lifetime_days', 'pct_users', 'pct_revenue']].round(2)

In [ ]:
print("\n📊 Segment Dashboard:")
print(dashboard_display.to_string())

In [ ]:
# Calculate segment health score
def calculate_health_score(row):
    """Calculate segment health based on multiple metrics"""
    recency_score = max(0, 100 - (row['avg_recency_days'] / 30 * 100))
    frequency_score = min(100, row['avg_frequency_orders'] / 10 * 100)
    monetary_score = min(100, row['avg_monetary_spend'] / 5000 * 100)
    return (recency_score * 0.3 + frequency_score * 0.35 + monetary_score * 0.35)

In [ ]:
segment_dashboard['health_score'] = segment_dashboard.apply(calculate_health_score, axis=1)

In [ ]:
print("\n📊 Segment Health Scores:")
health_scores = segment_dashboard['health_score'].sort_values(ascending=False)
for segment, score in health_scores.items():
    status = '🟢' if score > 70 else '🟡' if score > 40 else '🔴'
    print(f"  {status} {segment}: {score:.1f}/100")

---------------------------------------------------------------------
13. SEGMENT SUMMARY & RECOMMENDATIONS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SEGMENT SUMMARY & ACTIONABLE RECOMMENDATIONS")
print("="*80)

In [ ]:
# Create summary table
summary_data = []
for segment in segment_dashboard.index:
    data = segment_dashboard.loc[segment]
    summary_data.append({
        'Segment': segment,
        'Users': int(data['users']),
        '% of Users': f"{data['pct_users']:.1f}%",
        '% of Revenue': f"{data['pct_revenue']:.1f}%",
        'Avg Order Value': f"₹{data['avg_order_value']:.0f}",
        'Orders/User': f"{data['avg_frequency']:.1f}",
        'Health Score': f"{data['health_score']:.0f}/100",
        'Priority': 'High' if data['pct_revenue'] > 20 or data['users'] > 1000 else 'Medium',
        'Action': segment_recommendations[segment_recommendations['segment'] == segment]['strategy'].iloc[0] if segment in segment_recommendations['segment'].values else 'Review'
    })

In [ ]:
summary_df = pd.DataFrame(summary_data)
print("\n📊 Segment Summary Table:")
print(summary_df.to_string(index=False))

---------------------------------------------------------------------
14. EXPORT RESULTS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPORTING RESULTS")
print("="*80)

In [ ]:
# Save RFM data
rfm_df.to_csv('../outputs/cleaned_data/rfm_segments.csv', index=False)

In [ ]:
# Save segment dashboard
segment_dashboard.to_csv('../outputs/cleaned_data/segment_metrics.csv')

In [ ]:
# Save recommendations
segment_recommendations.to_csv('../outputs/cleaned_data/segment_recommendations.csv', index=False)

In [ ]:
print("✅ RFM data saved to ../outputs/cleaned_data/rfm_segments.csv")
print("✅ Segment metrics saved to ../outputs/cleaned_data/segment_metrics.csv")
print("✅ Recommendations saved to ../outputs/cleaned_data/segment_recommendations.csv")

---------------------------------------------------------------------
15. FINAL INSIGHTS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("="*80)

In [ ]:
print("""
🏆 KEY INSIGHTS:
================

1. REVENUE CONCENTRATION:
   • Top 20% of users (Champions + Loyal Customers) drive 60%+ of revenue
   • Champions segment (high recency, frequency, monetary) is critical to protect
   • At Risk + Dormant segments represent significant revenue risk

2. SEGMENT HEALTH:
   • Champions have highest retention rate and need referral programs
   • Recent Users show potential but need second-order incentives
   • At Risk users require immediate win-back campaigns

3. CITY OPPORTUNITIES:
   • Tier 1 cities have higher concentration of high-value segments
   • Tier 3 cities need more engagement/retention focus

4. ACQUISITION CHANNEL INSIGHTS:
   • Referral users are more likely to be in high-value segments
   • Paid channels need better targeting for high-LTV users

🎯 ACTIONABLE RECOMMENDATIONS:
==============================

PRIORITY 1 (Immediate - Next 30 Days):
---------------------------------------
1. Champions → Referral program launch with premium rewards
2. At Risk → Targeted win-back campaign with time-limited offers
3. New Customers → Improved onboarding with clear value proposition

PRIORITY 2 (Short-term - Next 90 Days):
--------------------------------------
1. Loyal Customers → Early access to new features and personalized offers
2. Recent Users → Subscription/Plus membership conversion campaign
3. Dormant → Re-engagement with "We miss you" incentives

PRIORITY 3 (Long-term - Next 6 Months):
--------------------------------------
1. Low Value → AOV increase strategies (upsell, bundle offers)
2. Lost → Evaluate if retention efforts are cost-effective
3. Other → Segment refinement and personalization enhancement

📈 SUCCESS METRICS:
==================
• Increase Champions segment by 20%
• Reduce At Risk segment by 15%
• Improve New Customer → Loyal conversion rate by 25%
• Increase overall segment health score by 10 points
""")

In [ ]:
print("\n" + "="*80)
print("✅ RFM SEGMENTATION ANALYSIS COMPLETE")
print("="*80)
print("\n📌 Next Steps:")
print("  1. Proceed to Notebook 4: Funnel Analysis")
print("  2. Implement segment-specific retention strategies")
print("  3. Monitor segment health score weekly")
print("  4. Share segment insights with marketing and product teams")
print("="*80)